# <font color="Green">**Notebook Purpose**</font>

This notebook constructs **eTable 11 (Patient Characteristics by Race/Ethnicity)** for the DOM T2D manuscript. It addresses Reviewer 5's request for baseline clinical profiles (BMI, HbA1c, HF, CKD) stratified by ethnic group.

**Structure:** Same row layout as the revised Table 1, but with race/ethnicity moved from a row dimension to the **column** dimension. The "Race/ethnicity" row block is removed (since it is the stratifier).

**Inputs:** Same as `Table1_Construction.ipynb`, including the `baseline_a1c.csv` and `baseline_bmi.csv` files produced upstream by `get_closest_continuous_value_after_t0` (pre-treatment, 183-day window within 2019). Early-disengagement patients are excluded.

**Output:** `eTable11_by_race.xlsx` — one row per characteristic, one column per race/ethnicity group plus Total.

## Section 1 — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl

In [ ]:
!pip install xlsxwriter

In [ ]:
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
patient_comorbidities = pd.read_csv('/content/patient_comorbidities.csv')
baseline_a1c = pd.read_csv('/content/baseline_a1c_all.csv')
baseline_bmi = pd.read_csv('/content/baseline_bmi_all.csv')

with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pkl.load(f)

patient_comorbidities['date'] = pd.to_datetime(patient_comorbidities['date'], errors='coerce')

## Section 2 — Exclude Early Dropouts

Apply the same dropout-exclusion logic as `Table1_Construction.ipynb` to land on the primary analytic cohort of 9,327 patients.

In [ ]:
dropout_ids = set()
for _, id_list in early_dropout_patients.items():
    dropout_ids.update([str(x) for x in id_list])

def exclude_dropouts(df, pid_col='patient_id'):
    out = df.copy()
    out[pid_col] = out[pid_col].astype(str)
    return out[~out[pid_col].isin(dropout_ids)]

dem = exclude_dropouts(patient_demographics)
a1c = exclude_dropouts(baseline_a1c)
bmi = exclude_dropouts(baseline_bmi)
com = exclude_dropouts(patient_comorbidities)

DENOMINATOR = len(dem)
print(f"Eligible patients (denominator): {DENOMINATOR:,}")

## Section 3 — Configuration

Bin definitions and race/ethnicity group setup. Bin edges and labels match `Table1_Construction.ipynb` exactly so eTable 11 is directly comparable to Table 1.

In [ ]:
PCT_DECIMALS = 1

# Race/ethnicity groups (input value -> display column name)
RACE_GROUPS = [
    ('White',    'White, NH'),
    ('Black',    'African American/Black, NH'),
    ('Hispanic', 'Hispanic/Latinx'),
    ('Asian',    'Asian, NH'),
    ('Other',    'Other, NH'),
]
TOTAL_COL = 'Total'
ALL_COLS  = [disp for _, disp in RACE_GROUPS] + [TOTAL_COL]

# Bin definitions (must match Table 1)
AGE_BINS   = [-np.inf, 45, 55, 65, np.inf]
AGE_LABELS = ['- 18-44', '- 45-54', '- 55-64', '- 65+']

A1C_BINS   = [-np.inf, 6.4, 8.0, 9.0, np.inf]
A1C_LABELS = ['- <6.4', '- 6.5-7.9', '- 8.0-8.9', '- >= 9.0']

BMI_BINS   = [-np.inf, 24.9, 29.9, 34.9, 39.9, np.inf]
BMI_LABELS = ['- <24.9', '- 25.0-29.9', '- 30.0-34.9', '- 35.0-39.9', '- >= 40']

WINDOW_END = pd.Timestamp('2019-06-01')

# ---- Patient ID sets for each column ----
dem['patient_id'] = dem['patient_id'].astype(str)
race_series = dem['race/ethnicity'].astype(str).str.strip()

col_pids = {}
for race_in, disp in RACE_GROUPS:
    col_pids[disp] = set(dem.loc[race_series == race_in, 'patient_id'])
col_pids[TOTAL_COL] = set(dem['patient_id'])

col_ns = {col: len(col_pids[col]) for col in ALL_COLS}
print('Column Ns:')
for col in ALL_COLS:
    print(f'  {col}: {col_ns[col]:,}')

def fmt(n, denom):
    if denom == 0:
        return '0 (0.0%)'
    return f'{int(n):,} ({100*n/denom:.{PCT_DECIMALS}f}%)'

## Section 4 — Build the Stratified Table

For each row, compute counts within each race column (and Total) using set intersection on patient IDs. This keeps the logic identical across columns and avoids label collisions on duplicated row labels (e.g., the two "- Missing" rows for HbA1c and BMI).

In [ ]:
rows = []

def add_header(label):
    rows.append({'Characteristic': label, **{col: '' for col in ALL_COLS}})

def add_row_from_pid_set(label, target_pids):
    """Compute |target_pids & col_pids[col]| / col_ns[col] for each column."""
    vals = {col: fmt(len(target_pids & col_pids[col]), col_ns[col]) for col in ALL_COLS}
    rows.append({'Characteristic': label, **vals})

# ---- Demographics ----
add_header('Demographics')
female_pids = set(dem.loc[
    dem['sex'].astype(str).str.upper().str.strip().isin(['F', 'FEMALE']),
    'patient_id'
])
add_row_from_pid_set('- Female sex', female_pids)

# ---- Age category ----
add_header('Age category, years')
dem['age_2019'] = 2019 - pd.to_numeric(dem['year_of_birth'], errors='coerce')
age_cuts = pd.cut(dem['age_2019'], bins=AGE_BINS,
                  labels=[l.replace('- ', '') for l in AGE_LABELS], right=False)
for edge_lbl, display_lbl in zip(age_cuts.cat.categories, AGE_LABELS):
    age_pids = set(dem.loc[age_cuts == edge_lbl, 'patient_id'])
    add_row_from_pid_set(display_lbl, age_pids)

# ---- Baseline HbA1c (pre-treatment, 2019) ----
add_header('Baseline HbA1c (pre-treatment, 2019)')
a1c['patient_id'] = a1c['patient_id'].astype(str)
a1c_vals = pd.to_numeric(a1c['baseline_a1c_value'], errors='coerce')
a1c_with_value = a1c.loc[a1c_vals.notna()].copy()
a1c_with_value['_val'] = pd.to_numeric(a1c_with_value['baseline_a1c_value'], errors='coerce')
a1c_binned = pd.cut(a1c_with_value['_val'], bins=A1C_BINS,
                    labels=[l.replace('- ', '') for l in A1C_LABELS], right=False)

for edge_lbl, display_lbl in zip(a1c_binned.cat.categories, A1C_LABELS):
    bin_pids = set(a1c_with_value.loc[a1c_binned == edge_lbl, 'patient_id'])
    add_row_from_pid_set(display_lbl, bin_pids)

# Missing = cohort - any baseline A1c value
a1c_has_value_pids = set(a1c_with_value['patient_id'])
a1c_missing_pids = col_pids[TOTAL_COL] - a1c_has_value_pids
add_row_from_pid_set('- Missing', a1c_missing_pids)

# ---- Baseline BMI (pre-treatment, 2019) ----
add_header('Baseline BMI (pre-treatment, 2019)')
bmi['patient_id'] = bmi['patient_id'].astype(str)
bmi_vals = pd.to_numeric(bmi['baseline_bmi_value'], errors='coerce')
bmi_with_value = bmi.loc[bmi_vals.notna()].copy()
bmi_with_value['_val'] = pd.to_numeric(bmi_with_value['baseline_bmi_value'], errors='coerce')
bmi_binned = pd.cut(bmi_with_value['_val'], bins=BMI_BINS,
                    labels=[l.replace('- ', '') for l in BMI_LABELS], right=False)

for edge_lbl, display_lbl in zip(bmi_binned.cat.categories, BMI_LABELS):
    bin_pids = set(bmi_with_value.loc[bmi_binned == edge_lbl, 'patient_id'])
    add_row_from_pid_set(display_lbl, bin_pids)

bmi_has_value_pids = set(bmi_with_value['patient_id'])
bmi_missing_pids = col_pids[TOTAL_COL] - bmi_has_value_pids
add_row_from_pid_set('- Missing', bmi_missing_pids)

# ---- Comorbidities (before June 1, 2019) ----
add_header('Comorbidities before June 1, 2019')
com['patient_id'] = com['patient_id'].astype(str)
window = com[com['date'] <= WINDOW_END].copy()
for col_name in ['HF', 'CKD']:
    window[col_name] = window[col_name].astype(str).str.strip().str.upper().isin(
        ['TRUE', '1', 'T', 'Y', 'YES']
    )

hf_pids  = set(window.loc[window['HF'],  'patient_id'].unique())
ckd_pids = set(window.loc[window['CKD'], 'patient_id'].unique())
add_row_from_pid_set('- Heart Failure',         hf_pids)
add_row_from_pid_set('- Chronic Kidney Disease', ckd_pids)

# ---- Assemble DataFrame ----
table_df = pd.DataFrame(rows)

# Build display column headers like Table 1: "White, NH (N=5,686)"
col_rename = {col: f"{col} (N={col_ns[col]:,})" for col in ALL_COLS}
display_df = table_df.rename(columns=col_rename)

display_df

## Section 5 — Export to Excel

In [ ]:
OUT_XLSX = '/content/eTable11_by_race.xlsx'

with pd.ExcelWriter(OUT_XLSX, engine='xlsxwriter') as writer:
    sheet = 'eTable 11'
    startrow = 2

    display_df.to_excel(writer, sheet_name=sheet, index=False, startrow=startrow)

    wb = writer.book
    ws = writer.sheets[sheet]

    title_fmt  = wb.add_format({'bold': True, 'font_size': 14, 'align': 'left'})
    header_fmt = wb.add_format({'bold': True, 'font_size': 11, 'bottom': 1, 'text_wrap': True, 'valign': 'top'})
    body_fmt   = wb.add_format({'font_size': 11, 'valign': 'top'})

    ws.merge_range(0, 0, 0, len(display_df.columns) - 1,
                   'eTable 11. Patient Characteristics by Race/Ethnicity', title_fmt)

    for ci, col in enumerate(display_df.columns):
        ws.write(startrow, ci, col, header_fmt)

    ws.set_column(0, 0, 42, body_fmt)
    ws.set_column(1, len(display_df.columns) - 1, 24, body_fmt)
    ws.set_row(startrow, 36)

print(f'Wrote {OUT_XLSX}')

## Section 6 — Suggested Footnote and Discussion Sentence

**Suggested eTable 11 footnote:**

> Values are summarized as N (%) within each race/ethnicity column. Percentages use each column's own denominator. Baseline glycated hemoglobin (HbA1c) and body mass index (BMI) represent the closest recorded value at or before each patient's first glucose-lowering medication (GLM) prescription within 2019, restricted to a 183-day pre-treatment window. Patients without a qualifying pre-treatment value are reported as missing. Comorbidities indicate documented heart failure (HF) or chronic kidney disease (CKD) between January 1 and June 1, 2019. Race and ethnicity categories reflect self-reported race as recorded in the electronic health record. **Abbreviations:** NH, Non-Hispanic.

**Reminder:** Once eTable 11 is produced, populate the Discussion sentence in the response letter (currently a placeholder) with two or three concise descriptive observations drawn from the table — e.g., which ethnic groups had higher/lower baseline HbA1c, which had higher HF/CKD prevalence, and any notable BMI distribution differences. Keep the framing descriptive, not causal, consistent with R1C7.